# Reinforcement Learning — Task 1: Tabular Q-Learning
### Environment: Gymnasium `Taxi` (v4 with automatic fallback to v3)

**Author:** Senior ML Engineer / RL Specialist  
**Framework:** `gymnasium` (the maintained successor to the deprecated `gym`)  
**Algorithm:** Off-policy, tabular **Q-Learning** with an $\epsilon$-greedy behaviour policy and $\epsilon$-decay.

---

## 1. Introduction & Justification

### Why the *Taxi* environment?

For a **tabular** Q-Learning study we need a problem whose state and action spaces are **finite and discrete**, so that the action-value function $Q(s,a)$ can be stored *exactly* in a lookup table — no function approximation, no neural network, no hidden bias from a feature encoder. *Taxi* satisfies this perfectly:

| Property | Taxi | Why it matters for tabular Q-Learning |
|---|---|---|
| **State space** | `Discrete(500)` | Small enough to hold the full $500 \times 6$ Q-table in memory, large enough to be non-trivial. |
| **Action space** | `Discrete(6)` | A clean, well-defined set of primitive actions (move N/S/E/W, pickup, dropoff). |
| **MDP structure** | Fully observable, Markovian | The next state and reward depend only on the current `(state, action)` — the core assumption Q-Learning is built on. |
| **Reward shaping** | Built-in, sparse-ish | `-1` per step, `+20` for a correct drop-off, `-10` for an illegal pickup/drop-off — forces the agent to learn an *efficient* route, not just any route. |

### Why *not* the usual tutorial clichés?

The most over-used tabular examples are **`FrozenLake`** (too small / often solved by luck on the slippery variant) and **`CliffWalking`** (essentially a single optimal path). Continuous classics like **`CartPole`** or **`MountainCar`** are *not* tabular at all — their observation is a real-valued vector, which would force discretisation or a neural network and defeat the purpose of this task.

*Taxi* hits the sweet spot: it has a **compositional task** (you must first *navigate* to the passenger, *pick up*, *navigate* to the destination, then *drop off*). This temporal structure means a naive agent earns large negative returns, and only a genuinely learned policy converges to consistently positive episode rewards — making the learning curve **visually convincing** rather than a flat line.

> **Note on the version.** The task brief specifies `Taxi-v4`. The publicly released `gymnasium` registry ships `Taxi-v3`. To be robust, the code below **requests `Taxi-v4` first and automatically falls back to `Taxi-v3`** if v4 is not registered in the installed build. The algorithm is identical either way.

## 2. Setup

Install/upgrade `gymnasium` and import the libraries. On Google Colab the `pip install` line below is all that is required.

In [ ]:
# Colab: install the maintained `gymnasium` (NOT the deprecated `gym`).
# `-q` keeps the output quiet. Safe to re-run; pip is idempotent.
!pip install -q "gymnasium>=0.29" matplotlib numpy

In [ ]:
import random
from collections import deque

import gymnasium as gym          # maintained successor to OpenAI `gym`
import numpy as np               # vectorised Q-table storage & math
import matplotlib.pyplot as plt  # learning-curve visualisation

# Reproducibility: fix every source of randomness we control.
# (Q-Learning is stochastic via epsilon-greedy and env transitions, so a
#  fixed seed makes the training curve and demo repeatable.)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('gymnasium version:', gym.__version__)

## 3. Environment Analysis

We create the environment and inspect its **observation space** and **action space**. Understanding these dimensions tells us the exact shape of the Q-table we must allocate: `(n_states, n_actions)`.

In [ ]:
def make_taxi_env(render_mode=None):
    """Create the Taxi environment.

    Tries `Taxi-v4` (as requested in the brief) and transparently falls
    back to `Taxi-v3` (the version shipped in the public gymnasium
    registry). Returns the env and the id that actually loaded.
    """
    for env_id in ('Taxi-v4', 'Taxi-v3'):
        try:
            env = gym.make(env_id, render_mode=render_mode)
            return env, env_id
        except gym.error.Error:
            # This version is not registered in the installed build;
            # try the next candidate.
            continue
    raise RuntimeError('Neither Taxi-v4 nor Taxi-v3 is available.')


# ANSI render_mode lets us print a text picture of the grid for the demo.
env, env_id = make_taxi_env(render_mode='ansi')
print(f'Loaded environment: {env_id}')
print('-' * 50)

# --- Observation (state) space -----------------------------------------
# Taxi encodes the FULL world state into a single integer:
#   ((taxi_row * 5 + taxi_col) * 5 + passenger_location) * 4 + destination
# -> 5 * 5 * 5 * 4 = 500 discrete states.
print('Observation space :', env.observation_space)
print('  -> type         :', type(env.observation_space).__name__)
print('  -> n states     :', env.observation_space.n)
print('-' * 50)

# --- Action space ------------------------------------------------------
# 0=move South, 1=move North, 2=move East, 3=move West,
# 4=Pickup passenger, 5=Drop-off passenger.
print('Action space      :', env.action_space)
print('  -> type         :', type(env.action_space).__name__)
print('  -> n actions    :', env.action_space.n)
print('-' * 50)

ACTION_MEANINGS = {
    0: 'South', 1: 'North', 2: 'East',
    3: 'West', 4: 'Pickup', 5: 'Drop-off',
}
print('Action meanings   :', ACTION_MEANINGS)

## 4. Q-Learning Agent (class-based)

The agent owns the **Q-table** and exposes three responsibilities:

1. **`choose_action`** — the $\epsilon$-greedy behaviour policy (explore vs. exploit).
2. **`update`** — the tabular Q-Learning Bellman update.
3. **`decay_epsilon`** — anneal exploration over time.

### The update rule implemented below

$$Q(s,a) \leftarrow (1-\alpha)\,Q(s,a) + \alpha\,\Big[R + \gamma\,\max_{a'} Q(s',a')\Big]$$

The bracketed term $R + \gamma\max_{a'}Q(s',a')$ is the **TD target** (a bootstrapped estimate of the optimal return), and $\alpha$ controls how far we move the old estimate towards that target.

In [ ]:
class QLearningAgent:
    """Tabular off-policy Q-Learning agent with epsilon-greedy control.

    Parameters
    ----------
    n_states, n_actions : int
        Dimensions of the discrete environment -> shape of the Q-table.
    alpha : float
        Learning rate (step size). How strongly a new experience
        overwrites the old Q estimate.
    gamma : float
        Discount factor. Weight placed on future vs. immediate reward.
    epsilon : float
        Initial probability of taking a RANDOM action (exploration).
    epsilon_min : float
        Floor below which epsilon never decays (keep a little exploration).
    epsilon_decay : float
        Multiplicative decay applied to epsilon after each episode.
    """

    def __init__(self, n_states, n_actions, alpha=0.10, gamma=0.99,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        # The Q-table: one row per state, one column per action.
        # Initialised to zeros (optimistic-neutral start). Shape (500, 6).
        self.q_table = np.zeros((n_states, n_actions), dtype=np.float64)

    def choose_action(self, state, greedy=False):
        """Select an action using the epsilon-greedy strategy.

        With probability epsilon -> EXPLORE (random action).
        Otherwise               -> EXPLOIT (best known action).
        Set `greedy=True` to force pure exploitation (used at test time).
        """
        # EXPLORATION branch: a uniformly random action lets the agent
        # discover state-action pairs it would never pick greedily, which
        # is what stops it getting stuck in a sub-optimal policy early on.
        if (not greedy) and (np.random.random() < self.epsilon):
            return np.random.randint(self.n_actions)

        # EXPLOITATION branch: choose the action with the highest Q-value
        # for this state. np.argmax breaks ties by taking the FIRST max,
        # which is deterministic and fine here.
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state, done):
        """Apply the tabular Q-Learning Bellman update."""
        # Current estimate Q(s, a) that we are about to refine.
        old_value = self.q_table[state, action]

        # Best achievable value from the NEXT state, max_a' Q(s', a').
        # This is the 'off-policy' part: we bootstrap on the GREEDY next
        # action regardless of what epsilon-greedy will actually do next.
        next_max = np.max(self.q_table[next_state])

        # (1 - done) zeroes out the future term on a TERMINAL transition.
        # Rationale: once the episode has ended there IS no s', so the
        # return is just the final reward R. Multiplying by (1 - done)
        # cleanly switches the bootstrap off without a separate `if`.
        td_target = reward + self.gamma * next_max * (1 - done)

        # Bellman update in the (1-alpha)*old + alpha*target form. This is
        # algebraically identical to old + alpha*(target - old); we use the
        # blended form exactly as written in the task specification.
        new_value = (1 - self.alpha) * old_value + self.alpha * td_target

        # Write the refined estimate back into the table.
        self.q_table[state, action] = new_value

    def decay_epsilon(self):
        """Reduce epsilon multiplicatively, never below epsilon_min."""
        self.epsilon = max(self.epsilon_min,
                           self.epsilon * self.epsilon_decay)

## 5. Training Loop

We train for **2,000 episodes**. Each episode:

1. Reset the environment to a random start state.
2. Loop: pick an $\epsilon$-greedy action, step, **update the Q-table**.
3. After the episode ends, **decay $\epsilon$** and record the total reward.

The Gymnasium step API returns five values — `obs, reward, terminated, truncated, info` — and an episode is over when `terminated or truncated`. We feed `terminated` (the true MDP terminal signal) into the `(1 - done)` logic of the Bellman update.

In [ ]:
# --- Hyperparameters ---------------------------------------------------
N_EPISODES = 2000      # total training episodes (per the brief)
MAX_STEPS = 200        # safety cap per episode (Taxi truncates at 200)

ALPHA = 0.10           # learning rate
GAMMA = 0.99           # discount factor (Taxi rewards future drop-off)
EPSILON_START = 1.0    # start fully exploratory
EPSILON_MIN = 0.01     # keep 1% exploration forever
EPSILON_DECAY = 0.9995 # ~ multiplicative anneal across 2000 episodes

# Re-create a fresh, SEEDED training env (no rendering -> faster).
train_env, _ = make_taxi_env(render_mode=None)

agent = QLearningAgent(
    n_states=train_env.observation_space.n,
    n_actions=train_env.action_space.n,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon=EPSILON_START,
    epsilon_min=EPSILON_MIN,
    epsilon_decay=EPSILON_DECAY,
)

episode_rewards = []        # total reward per episode (for the graph)
epsilon_history = []        # epsilon per episode (to visualise decay)
recent = deque(maxlen=100)  # rolling window for a live console readout

for episode in range(N_EPISODES):
    # Seed the reset on the FIRST episode only, so the whole run is
    # reproducible but subsequent episodes still vary their start state.
    if episode == 0:
        state, _info = train_env.reset(seed=SEED)
    else:
        state, _info = train_env.reset()

    total_reward = 0.0

    for _step in range(MAX_STEPS):
        # 1) Choose an action with the epsilon-greedy behaviour policy.
        action = agent.choose_action(state)

        # 2) Apply it. Gymnasium returns a 5-tuple.
        next_state, reward, terminated, truncated, _info = \
            train_env.step(action)

        # `terminated` = reached a real MDP terminal state (passenger
        #   dropped off correctly). This is the signal used in (1 - done).
        # `truncated`  = hit the time limit; the episode ends but the
        #   state is NOT terminal, so we do not zero the bootstrap on it.
        agent.update(state, action, reward, next_state, terminated)

        state = next_state
        total_reward += reward

        if terminated or truncated:
            break

    # 3) Anneal exploration and record metrics for this episode.
    agent.decay_epsilon()
    episode_rewards.append(total_reward)
    epsilon_history.append(agent.epsilon)
    recent.append(total_reward)

    # Lightweight progress print every 200 episodes.
    if (episode + 1) % 200 == 0:
        print(f'Episode {episode + 1:4d}/{N_EPISODES} | '
              f'avg(last100)={np.mean(recent):7.2f} | '
              f'epsilon={agent.epsilon:.3f}')

train_env.close()
print('\nTraining complete.')
print(f'Final 100-episode average reward: {np.mean(recent):.2f}')

## 6. Visualization — Total Reward vs. Episode

Raw per-episode reward is noisy (every episode starts in a different state). A **50-episode rolling average** reveals the underlying learning trend: the curve should climb from large negatives toward a stable, positive plateau (~ +7 to +9 for a well-trained Taxi agent). We overlay the $\epsilon$-decay on a secondary axis to show exploration shrinking as performance rises.

In [ ]:
def rolling_average(values, window):
    """Simple moving average; output aligned to the END of each window."""
    values = np.asarray(values, dtype=np.float64)
    if len(values) < window:
        return np.array([]), np.array([])
    # 'valid' mode -> one output per full window of `window` points.
    kernel = np.ones(window) / window
    avg = np.convolve(values, kernel, mode='valid')
    # x-positions: the average at index i covers episodes [i, i+window-1].
    x = np.arange(window - 1, len(values))
    return x, avg


WINDOW = 50
roll_x, roll_y = rolling_average(episode_rewards, WINDOW)

fig, ax1 = plt.subplots(figsize=(12, 6))

# Raw reward per episode (faint, for texture).
ax1.plot(episode_rewards, color='tab:blue', alpha=0.25, linewidth=0.8,
         label='Total reward (per episode)')

# 50-episode rolling average (the headline learning trend).
ax1.plot(roll_x, roll_y, color='tab:red', linewidth=2.5,
         label=f'Rolling average (window={WINDOW})')

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Total Reward', fontsize=12)
ax1.set_title('Q-Learning on Taxi: Total Reward vs. Episode',
              fontsize=14, fontweight='bold')
ax1.axhline(0, color='grey', linestyle='--', linewidth=1, alpha=0.7)
ax1.grid(True, alpha=0.3)

# Secondary axis: epsilon decay over training.
ax2 = ax1.twinx()
ax2.plot(epsilon_history, color='tab:green', linewidth=1.5,
         linestyle=':', label='Epsilon (exploration rate)')
ax2.set_ylabel('Epsilon', fontsize=12, color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')
ax2.set_ylim(0, 1.05)

# Merge legends from both axes into one box.
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right',
           fontsize=10)

plt.tight_layout()
plt.show()

## 7. Agent Demonstration — Watch the Trained Policy Play

We now let the **trained** agent play one full episode using a **pure greedy** policy (`greedy=True`, i.e. no exploration). With ANSI rendering we print each frame so you can watch the taxi navigate to the passenger, execute **Pickup**, drive to the destination, and execute **Drop-off** for the `+20` terminal reward.

In [ ]:
import time
from IPython.display import clear_output


def demonstrate(agent, max_steps=50, pause=0.4, clear=True):
    """Play ONE episode greedily, rendering each step as ANSI text."""
    # Fresh env with ANSI rendering so render() returns a printable string.
    demo_env, demo_id = make_taxi_env(render_mode='ansi')
    state, _info = demo_env.reset(seed=SEED + 1)  # reproducible demo run

    total_reward = 0.0
    frames = []

    for step in range(max_steps):
        # Pure exploitation: always take the best known action.
        action = agent.choose_action(state, greedy=True)
        state, reward, terminated, truncated, _info = \
            demo_env.step(action)
        total_reward += reward

        frame = demo_env.render()  # ANSI string of the current grid
        frames.append((step + 1, action, reward, total_reward, frame))

        if terminated or truncated:
            break

    demo_env.close()

    # Replay the frames with a short pause for a 'video' feel.
    for step_no, action, reward, cum, frame in frames:
        if clear:
            clear_output(wait=True)
        print(f'Environment: {demo_id}')
        print(f'Step {step_no:2d} | Action: {ACTION_MEANINGS[action]:>8} '
              f'| Reward: {reward:+.0f} | Cumulative: {cum:+.0f}')
        print(frame)
        time.sleep(pause)

    success = frames and frames[-1][2] == 20  # +20 == correct drop-off
    print('-' * 40)
    print(f'Episode finished in {len(frames)} steps. '
          f'Total reward: {total_reward:+.0f}')
    print('RESULT:', 'SUCCESS - passenger delivered!' if success
          else 'Did not deliver within step limit.')
    return total_reward


_ = demonstrate(agent, max_steps=50, pause=0.4, clear=True)

### Quick quantitative sanity check

One render is anecdotal. Below we evaluate the greedy policy over 100 episodes (no rendering) to confirm it solves the task **reliably**.

In [ ]:
def evaluate(agent, episodes=100):
    """Run the greedy policy for N episodes; return summary statistics."""
    eval_env, _ = make_taxi_env(render_mode=None)
    rewards, steps_taken, successes = [], [], 0

    for ep in range(episodes):
        state, _info = eval_env.reset(seed=SEED + 100 + ep)
        total, last_reward = 0.0, 0.0
        for step in range(200):
            action = agent.choose_action(state, greedy=True)
            state, reward, terminated, truncated, _info = \
                eval_env.step(action)
            total += reward
            last_reward = reward
            if terminated or truncated:
                break
        rewards.append(total)
        steps_taken.append(step + 1)
        successes += int(last_reward == 20)

    eval_env.close()
    print(f'Episodes evaluated : {episodes}')
    print(f'Success rate       : {successes / episodes:.1%}')
    print(f'Mean reward        : {np.mean(rewards):.2f} '
          f'(+/- {np.std(rewards):.2f})')
    print(f'Mean steps/episode : {np.mean(steps_taken):.1f}')


evaluate(agent, episodes=100)

## 8. Theoretical Explanation

### 8.1 Choice of hyperparameters ($\alpha$, $\gamma$, $\epsilon$)

| Symbol | Value | Role | Why this value |
|---|---|---|---|
| $\alpha$ (learning rate) | `0.10` | Step size of each Q update. | Taxi transitions are **deterministic**, so a moderate rate learns quickly without the table oscillating. Too high (→1.0) makes each update overwrite history; too low slows convergence past 2,000 episodes. |
| $\gamma$ (discount) | `0.99` | Weight on future reward. | The `+20` payoff arrives **only at the very end** of a multi-step task, so we need a high discount for that distant reward to propagate back and shape earlier navigation decisions. |
| $\epsilon$ (exploration) | `1.0 → 0.01` | Probability of a random action. | Start at `1.0` (know nothing → explore everything), decay multiplicatively by `0.9995`/episode toward a `0.01` floor so the agent increasingly **exploits** what it has learned while never fully stopping exploration. |

Over 2,000 episodes, $\epsilon = 1.0 \times 0.9995^{2000} \approx 0.37$ before the floor; in practice the `max(epsilon_min, ...)` clamp and the long tail mean exploration is already low while the table is still being polished — a healthy explore-then-exploit schedule.

### 8.2 How the `(1 - done)` logic was handled

The Bellman target is

$$\text{target} = R + \gamma \,\big(\max_{a'} Q(s',a')\big)\,(1-\text{done}).$$

When a transition is **terminal** (`done == 1`, the passenger is correctly dropped off), there is **no successor state** $s'$ — the episode is over, so its return is simply the immediate reward $R$. Multiplying the bootstrap term by `(1 - done)` zeroes $\gamma\max_{a'}Q(s',a')$ on exactly those transitions, preventing the agent from adding a phantom future value to a state that has none. On **non-terminal** steps `done == 0`, so `(1 - done) == 1` and the full bootstrap is kept.

Crucially, we pass **`terminated`** (not `terminated or truncated`) into this flag. A `truncated` episode hit the 200-step time limit but its final state is *not* a true MDP terminal — bootstrapping there is still correct, so it must **not** be masked.

### 8.3 Exploration vs. Exploitation in this code

These two behaviours live in `QLearningAgent.choose_action`:

```python
if (not greedy) and (np.random.random() < self.epsilon):
    return np.random.randint(self.n_actions)   # EXPLORATION
return int(np.argmax(self.q_table[state]))     # EXPLOITATION
```

* **Exploration** (the `if` branch): with probability $\epsilon$ the agent ignores its current knowledge and picks a **uniformly random** action. This is what lets it *discover* the reward structure — e.g. stumbling onto the `Pickup`/`Drop-off` actions in the right cells — rather than greedily repeating whatever looked best from a near-empty Q-table.
* **Exploitation** (the `argmax`): the agent trusts its Q-table and takes the **highest-value** action for the current state, i.e. it *uses* what it has learned to maximise reward.

The fundamental tension is the **explore/exploit trade-off**: explore too little and the agent locks into a sub-optimal route it found early; explore too much and it never cashes in on its knowledge. $\epsilon$-decay resolves this over time — heavy exploration when the table is ignorant, near-pure exploitation once it is well-formed. At **test time** we pass `greedy=True`, disabling exploration entirely to measure the learned policy on its own merits.

### 8.4 Why Q-Learning is the right algorithm here

Q-Learning is **off-policy** and **model-free**: it learns the optimal action-value function $Q^*$ directly from sampled transitions, bootstrapping on the *greedy* next action ($\max_{a'}$) even while it *behaves* $\epsilon$-greedily. Because Taxi is a small, discrete, fully observable MDP, the tabular form is **guaranteed to converge** to the optimal policy given sufficient visits to every state-action pair — which the $\epsilon$-greedy exploration schedule ensures.